# 推理服务与量化补充线 · 第 5/8 课：推理并行与 Prefill/Decode 解耦

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：根据 prefill/decode 独立服务率计算最小副本数，并解释 TP、DP 与 P/D 解耦的边界。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

`train/` 已讲训练 TP/PP；推理更关心权重是否装下、KV 容量、TTFT/ITL 和两阶段资源形态，不沿用训练最优配置。

前置：train 第 1～5 课、CUDA/Triton 基础、Transformer attention。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

TP 把单模型计算/权重分到多卡；DP 复制模型服务不同请求；P/D 解耦让 prefill 节点和 decode 节点独立扩缩，并传输 KV。

### 数据与控制如何流动

请求先按路由进入某个 prefill replica，完成后把 KV 与请求状态交给兼容的 decode replica；容量规划分别以 prompt-token/s 和 output-token/s 建账，再检查跨阶段队列与传输。

### 正确性条件与常见误区

每阶段长期到达负载必须低于服务能力并保留余量；P/D 解耦不能忽略 KV 传输兼容性、失败重试和请求粘性。它通常用于隔离 SLO，不保证总吞吐上升。

### 性能、成本与工程取舍

TP 增大可装下模型、缩短单次计算，却引入每层通信；DP 提升总容量但复制权重；解耦提高资源匹配和尾延迟控制，却新增网络与运营复杂度。

## 具体演示

到达率 20 req/s，平均 prompt 1000 tokens、输出 100 tokens：prefill 需求 20k tok/s，decode 需求 2k tok/s。若单副本能力分别 8k/700 tok/s，至少需 3 个 P 和 3 个 D（未计余量）。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐两阶段独立容量规划；utilization_target 必须在 (0,1]。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
import math

def pd_replicas(arrival_rps, prompt_tokens, output_tokens,
                prefill_tps_per_replica, decode_tps_per_replica,
                utilization_target=0.8):
    if not 0 < utilization_target <= 1:
        raise ValueError("invalid utilization target")
    prefill_demand = arrival_rps * prompt_tokens
    decode_demand = arrival_rps * output_tokens
    # TODO：按目标利用率折减每副本可用能力并向上取整。
    return ______

assert pd_replicas(20, 1000, 100, 8000, 700, 0.8) == (4, 4)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么训练时 TP=8 合理，不代表在线 decode 也应 TP=8？

**你的答案：**


### Q2

P/D 解耦为何主要改善 SLO 隔离，而不应默认声称提高吞吐？

**你的答案：**


### Q3

解耦部署中 decode 节点故障，重试为何不能只把请求发到任意 D？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
import math

def pd_replicas(arrival_rps, prompt_tokens, output_tokens,
                prefill_tps_per_replica, decode_tps_per_replica,
                utilization_target=0.8):
    if not 0 < utilization_target <= 1:
        raise ValueError("invalid utilization target")
    prefill_demand = arrival_rps * prompt_tokens
    decode_demand = arrival_rps * output_tokens
    p = math.ceil(prefill_demand / (prefill_tps_per_replica * utilization_target))
    d = math.ceil(decode_demand / (decode_tps_per_replica * utilization_target))
    return p, d

assert pd_replicas(20, 1000, 100, 8000, 700, 0.8) == (4, 4)


### Q1 参考答案

训练的大 batch 可摊通信并需要分片状态；decode M 小，每层 TP collective 可能主导延迟。如果模型和 KV 能装入更少 GPU，较小 TP 加更多 DP 副本可能有更高容量和更低 ITL。必须按目标 shape 实测。

### Q2 参考答案

它把计算型 prefill 和带宽/KV 型 decode 分开，减少互相干扰并可独立扩容；但 KV 传输、额外排队和网络成本可能抵消收益，总算力并未凭空增加。

### Q3 参考答案

新 D 需要对应请求的完整 KV 或必须重做 prefill；还要保证模型/adapter/tokenizer 版本一致和流式响应去重。需要明确 KV 持久性、会话路由和幂等协议。

## 参考资料

- [vLLM disaggregated serving](https://docs.vllm.ai/en/stable/examples/disaggregated/disaggregated_serving/)
- [vLLM serving documentation](https://docs.vllm.ai/en/latest/cli/serve/)
- [TensorRT-LLM documentation](https://nvidia.github.io/TensorRT-LLM/)

API 与平台能力会演进；部署前应按目标版本重新核对。